In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
# Load data set
# cleaned data
data = pd.read_csv(r"..\..\..\Datasets\For analysis\full_data_CIFAR100_v3.csv")
# used to generate Results
#data = pd.read_csv(r"data.csv")

# Ensure we only have data with 80/20 datasplit
split_07 = [3911, 3912, 3913, 3914, 3915, 3916, 3917, 3918, 3919, 39110, 39111, 39112, 39113, 39114, 39115]
split_09 = [3901,3902,3903,3904,3905,3906,3907,3908,3909,39010,39011,39012,39013,39014,39015]
split = split_07 + split_09
data = data[~data["exp_id"].isin(split)]

# Rename names for readbility
rename_map = {
    "vit_l_32": "ViT-L/32",
    "vit_b_16": "ViT-B/16",
    "vit_b_32": "ViT-B/32",
    "vit_small_patch16_224": "ViT-S/16",
    "vit_small_patch32_224": "ViT-S/32",
    "vit_tiny_patch16_224": "ViT-T/16",
}

data["model"] = data["model"].replace(rename_map)

In [ ]:
# Functions to analyse
def make_subset(df, lr, dropout, batch_size):
    model_list = ["ViT-B/32", "ViT-S/32", "ViT-T/16", "ViT-S/16", "ViT-B/16","ViT-L/32"]
    # Freeze other hyperparameters so can get all configurations when changing batch rate while freezing other parameters
    subset = df[
        (df["batch_size"] == batch_size) &
        (df["lr"] == lr) &
        (df["model"].isin(model_list)) &
        (df["dropout"] == dropout) & 
        (df["weight_decay"] == 0.0)
        
    ]
    return subset

def rank(results_df, metric, ascending = False):
    # Rank batch sizes on a given metric, ie "accuracy" or "SPJ"
    df = results_df
    return (
        df.groupby("model")[metric]
        .mean()
        .sort_values(ascending=ascending)
        .reset_index()
        .rename(columns={metric: f"{metric}"})
    )

def summary_table(subset):
    # Summary table that takes mean of every configuration
    view = (
        subset
        .groupby(["model", "batch_size", "lr", "dropout"])
        .agg(
            accuracy=("accuracy", "mean"),
            precision=("precision", "mean"),
            recall=("recall", "mean"),
            specificity=("specificity", "mean"),
            energy=("total_energy_J", "mean"),
            energy_std=("total_energy_J", "std"),
            n_seeds=("accuracy", "nunique"),
            Eg=("Eg","mean"),
            Pg=("Pg", "mean"),
            FPJ=("FPJ", "mean"),
            EDPinv=("EDPinv", "mean"),
            SPJ=("SPJ", "mean"),
        )
        .reset_index()
    )
    print("=========================================")
    print("View for plotting")
    print(view)
    print("=========================================")
    return view



# ViT-b-32


In [ ]:
bs_to_assess = [128, 64, 256, 96, 156, 72]
vit_b_32 = make_subset(data,"ViT-B/32",0.03,0.00,bs_to_assess)
vit_b_32 = summary_table(vit_b_32)


vit_b_32 = vit_b_32.dropna()
selected = vit_b_32
print("ViT-B/32")  
Eg = rank(selected,"Eg", ascending=False)
Pg = rank(selected, "Pg", ascending=False)
ac = rank(selected, "accuracy", ascending=False)
re = rank(selected, "recall", ascending=False)
sp = rank(selected, "specificity", ascending=False)
pre = rank(selected, "precision", ascending=False)
fpj = rank(selected, "FPJ", ascending=False)
edpinv = rank(selected, "EDPinv", ascending=False)
spj = rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = rank(selected,"EgPg", ascending=False)
energy = rank(selected,"energy", ascending=True)

print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))